# Assignment 1

In [29]:
import time # measure how long an algo takes
import tracemalloc # measure m/m usage
import random 

### Bubble Sort

In [30]:
def bubble_sort(arr):
    n = len(arr)
    
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    
    return arr

### Merge Sort

In [31]:
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    
    mid = len(arr) // 2
    
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    
    result = []
    i = j = 0
    
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    
    result.extend(left[i:])
    result.extend(right[j:])
    
    return result

### Quick Sort

In [32]:
def quick_sort(arr):
    if len(arr) <= 1:
        return arr
    
    pivot = arr[len(arr) // 2]
    
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

#### Created input-data generator

In [33]:
def generate_datasets(size):
    random_data = [random.randint(1, 100000) for i in range(size)]
    sorted_data = sorted(random_data)
    reverse_sorted_data = sorted(random_data, reverse=True)
    return sorted_data, reverse_sorted_data, random_data

#### performance measurement function

In [37]:
def measure_time(sort_function, data):
    data_copy = data.copy()

    start_time = time.perf_counter()
    sort_function(data_copy)
    end_time = time.perf_counter()

    return end_time - start_time

In [38]:
def measure_memory(sort_function, data):
    data_copy = data.copy()

    tracemalloc.start()

    sort_function(data_copy)

    current_memory, peak_memory = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return peak_memory

### Execution Time

In [40]:
import pandas as pd

sizes = [100, 500, 1000, 5000, 10000]

algorithms = {
    "Bubble Sort": bubble_sort,
    "Merge Sort": merge_sort,
    "Quick Sort": quick_sort
}

results = []

for size in sizes:
    print(f"Testing input size: {size}")

    sorted_data, reverse_data, random_data = generate_datasets(size)

    datasets = {
        "Sorted": sorted_data,
        "Reverse-Sorted": reverse_data,
        "Random": random_data
    }

    for condition, data in datasets.items():

        for algorithm_name, algorithm in algorithms.items():

            time_taken = measure_time(algorithm, data)

            results.append({
                "Algorithm": algorithm_name,
                "Input Size": size,
                "Condition": condition,
                "Execution Time (seconds)": time_taken
            })

results_df = pd.DataFrame(results)

print("Experiment completed!")

Testing input size: 100
Testing input size: 500
Testing input size: 1000
Testing input size: 5000
Testing input size: 10000
Experiment completed!


In [41]:
results_df

,Algorithm,Input Size,Condition,Execution Time (seconds)
0,Bubble Sort,100,Sorted,0.001275
1,Merge Sort,100,Sorted,0.003030
2,Quick Sort,100,Sorted,0.002158
3,Bubble Sort,100,Reverse-Sorted,0.002147
4,Merge Sort,100,Reverse-Sorted,0.002436
5,Quick Sort,100,Reverse-Sorted,0.001645
6,Bubble Sort,100,Random,0.001710
7,Merge Sort,100,Random,0.002480
8,Quick Sort,100,Random,0.006760
9,Bubble Sort,500,Sorted,0.172124


In [42]:
memory_results = []

for size in sizes:
    print(f"Measuring memory for input size: {size}")

    sorted_data, reverse_data, random_data = generate_datasets(size)

    datasets = {
        "Sorted": sorted_data,
        "Reverse-Sorted": reverse_data,
        "Random": random_data
    }

    for condition, data in datasets.items():
        for algorithm_name, algorithm in algorithms.items():

            # Skip very expensive quadratic cases at size 10000
            if size == 10000 and algorithm_name in ["Bubble Sort", "Insertion Sort"]:
                continue

            peak_memory = measure_memory(algorithm, data)

            memory_results.append({
                "Algorithm": algorithm_name,
                "Input Size": size,
                "Condition": condition,
                "Peak Memory (bytes)": peak_memory
            })

print("Memory experiment completed!")

Measuring memory for input size: 100
Measuring memory for input size: 500
Measuring memory for input size: 1000
Measuring memory for input size: 5000
Measuring memory for input size: 10000
Memory experiment completed!


In [43]:
memory_df = pd.DataFrame(memory_results)
memory_df

,Algorithm,Input Size,Condition,Peak Memory (bytes)
0,Bubble Sort,100,Sorted,31834343
1,Merge Sort,100,Sorted,2032
2,Quick Sort,100,Sorted,2720
3,Bubble Sort,100,Reverse-Sorted,152768
4,Merge Sort,100,Reverse-Sorted,2032
5,Quick Sort,100,Reverse-Sorted,2808
6,Bubble Sort,100,Random,112
7,Merge Sort,100,Random,1824
8,Quick Sort,100,Random,5000
9,Bubble Sort,500,Sorted,236


In [44]:
results_df.to_csv("sorting_time_results.csv", index=False)
memory_df.to_csv("sorting_memory_results.csv", index=False)

print("Sorting results saved successfully!")

Sorting results saved successfully!


In [45]:
import gc

def measure_memory_clean(sort_function, data):
    gc.collect()
    
    data_copy = data.copy()
    
    tracemalloc.start()
    
    sort_function(data_copy)
    
    current_memory, peak_memory = tracemalloc.get_traced_memory()
    
    tracemalloc.stop()
    
    return peak_memory

In [46]:
sorted_data, reverse_data, random_data = generate_datasets(100)

memory = measure_memory_clean(bubble_sort, sorted_data)

print("Bubble Sort memory for sorted data (n=100):", memory, "bytes")

Bubble Sort memory for sorted data (n=100): 112 bytes


In [47]:
memory_results_clean = []

for size in sizes:
    print(f"Measuring memory for input size: {size}")

    sorted_data, reverse_data, random_data = generate_datasets(size)

    datasets = {
        "Sorted": sorted_data,
        "Reverse-Sorted": reverse_data,
        "Random": random_data
    }

    for condition, data in datasets.items():

        for algorithm_name, algorithm in algorithms.items():

            # Skip Bubble Sort at 10000 because it is extremely slow
            if size == 10000 and algorithm_name == "Bubble Sort":
                continue

            peak_memory = measure_memory_clean(algorithm, data)

            memory_results_clean.append({
                "Algorithm": algorithm_name,
                "Input Size": size,
                "Condition": condition,
                "Peak Memory (bytes)": peak_memory
            })

print("Clean memory experiment completed!")

Measuring memory for input size: 100
Measuring memory for input size: 500
Measuring memory for input size: 1000
Measuring memory for input size: 5000
Measuring memory for input size: 10000
Clean memory experiment completed!


In [48]:
memory_df = pd.DataFrame(memory_results_clean)

memory_df.to_csv("sorting_memory_results.csv", index=False)

print("Clean memory data saved successfully!")

Clean memory data saved successfully!
